# 06. Final training with 3-fold CV, inference and saved results

**What this notebook does:** trains the final model (all decisions from notebooks 00-05 in `nerkit/decisions.py`) once per frozen CV fold (0, 1, 2), scores each held-out fold, predicts the competition test set with each fold model, and saves everything so it can be reused: logits for every split, tokenizer, model weights, scores and one Kaggle-format CSV per fold.

**Settings:** GRU + CRF, word tokens with frozen fastText aligned vectors, 100k rows per language, structure tokens, early stopping on a 5% inner validation split, seed 3012026. Decoding: lenient merge, empty strings dropped, CRF bias 0.

**Where to run:** Colab with a GPU (Runtime > Change runtime type > GPU). It also runs locally; then it just takes longer. A finished fold is skipped on rerun, so a disconnected session resumes where it stopped.

**Before the first run on Colab**, put these into your Google Drive:
- the project folder `ai301/deep-learning-assignment3` (cloned automatically from the `model` branch on GitHub if missing; `model/` has to be pushed there first), and
- the four Kaggle files (`uk_geo_dataset.csv`, `ru_geo_dataset.csv`, `test.csv`, `labeling_sample.csv`) in `MyDrive/ai301/data/kse-ua-location-extraction-2026/`.

Everything the notebook writes goes to `MyDrive/ai301/ner_work/` on Colab, so it survives a disconnect.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ai301")
    ROOT = PROJECT / "deep-learning-assignment3"
    BRANCH = "model"  # the branch that holds model/; main does not have it
    if not ROOT.exists():
        PROJECT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/koalla05/deep-learning-assignment3.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "checkout", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=ROOT, check=True)
    subprocess.run(["git", "lfs", "install"], cwd=ROOT)
    subprocess.run(["git", "lfs", "pull"], cwd=ROOT)  # the frozen split manifests are stored with Git LFS
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightning", "pytorch-crf", "tokenizers"], check=True)
    os.environ["NER_DATA_DIR"] = str(PROJECT / "data" / "kse-ua-location-extraction-2026")
    os.environ["NER_WORK_DIR"] = str(PROJECT / "ner_work")
else:
    ROOT = Path.cwd()
    while not (ROOT / "eda").exists():
        ROOT = ROOT.parent

import torch

os.environ.setdefault("NER_WANDB", "off")
os.environ["NER_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
print("Device:", os.environ["NER_DEVICE"], "| project:", ROOT)

In [ ]:
import json
import shutil

import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, write_submission
from nerkit.config import ARTIFACTS_DIR, WORK_DIR
from nerkit.data import build_cache, load_test
from nerkit.decisions import DECISIONS, DECODING, base_config
from nerkit.experiment import read_results, run_experiment, select_training
from nerkit.metrics import evaluate

NOTEBOOK = "06_final_cv_training"
FOLDS = (0, 1, 2)
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}

build_cache()  # joins the CSV files with the frozen manifests; skipped when already cached
print("Work folder:", WORK_DIR)
print(json.dumps(DECISIONS, indent=1))

## Train one model per fold

Fold `k` trains on the other two folds (plus the `extra` training rows) and is scored on fold `k`, which the model never sees. The 5% inner split is used only for early stopping.

In [ ]:
records = [run_experiment(f"fold{fold}", base_config(fold=fold), NOTEBOOK) for fold in FOLDS]

## Cross-validation scores

Re-decoded from the saved held-out logits with the final decoding settings, so the scores can be reproduced from the files alone.

In [ ]:
rows_by_fold, scores = {}, []
for record in records:
    fold = record["config"]["fold"]
    _, _, _, heldout = select_training(base_config(fold=fold))
    logits, meta = load_logits(record["key"], "heldout")
    result = evaluate(heldout, decode_split(logits, heldout, crf=meta["crf"], **DECODE))
    scores.append(dict(fold=fold, mix_f1=result["mix"]["f1"], precision=result["mix"]["precision"],
                       recall=result["mix"]["recall"], uk_f1=result["uk"]["f1"], ru_f1=result["ru"]["f1"],
                       posts_f1=record["scores"]["posts"]["mix"]["f1"],
                       epochs=len(record["history"]), minutes=record["total_seconds"] / 60))
table = pd.DataFrame(scores).set_index("fold")
display(table.round(3))
cv = pd.DataFrame({"mean": table.mean(), "std": table.std()}).round(3)
display(cv)

## Test predictions

Every fold model predicts the 477 competition posts. One CSV per fold is written, so they can be submitted separately; combining them is left to the ensembling step. The logits of all splits are already saved next to each model.

In [ ]:
test_rows = load_test()
submissions = {}
for record in records:
    fold = record["config"]["fold"]
    logits, meta = load_logits(record["key"], "test")
    predictions = decode_split(logits, test_rows, crf=meta["crf"], **DECODE)
    path = write_submission(WORK_DIR / "submissions" / f"final_fold{fold}.csv", test_rows, predictions)
    submissions[fold] = path
    print(f"fold {fold}: {sum(bool(p) for p in predictions)} of {len(predictions)} posts have locations, "
          f"{sum(len(p) for p in predictions)} locations in total -> {path.name}")

## Everything saved for reuse

`final_cv/` collects what is needed to ensemble or to run the models again: per fold the logits for `inner`, `heldout`, `posts`, `labeling` and `test` (float16, one row per token, order O/B/I, CRF transition scores in the metadata), the tokenizer, the model weights, plus `summary.json` with the configuration, the CV scores and the decoding settings. `final_cv.zip` is the same folder for downloading.

In [ ]:
package = WORK_DIR / "final_cv"
package.mkdir(parents=True, exist_ok=True)
for record in records:
    fold = record["config"]["fold"]
    shutil.copytree(ARTIFACTS_DIR / record["key"], package / f"fold{fold}", dirs_exist_ok=True)
summary = dict(
    decisions=DECISIONS, decoding=DECODE, seed=records[0]["config"]["seed"],
    folds={f"fold{r['config']['fold']}": dict(run_key=r["key"], scores=r["scores"], epochs=len(r["history"]),
                                              training_rows=r["training_rows"], minutes=r["total_seconds"] / 60)
           for r in records},
    cv_mean=table.mean().to_dict(), cv_std=table.std().to_dict(),
    tag_order=["O", "B", "I"], note="Logits per split are in fold<k>/<split>.npz; see notebook 05 for the format.")
(package / "summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False), encoding="utf-8")
shutil.copytree(WORK_DIR / "submissions", package / "submissions", dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK_DIR / "final_cv"), "zip", package)
print("Saved:", package)
print("Archive:", archive, f"({Path(archive).stat().st_size / 1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(archive)